In [17]:
'''
The following code concatenates the original source files, create a multiindex dataframe, 
preprocesses, scale and standardize the data. 
The final result is stored into dataset_full.csv: 1555 stocks, 21 features, 40 quarters, 3 classes
Seed = 123

WARNING: the code takes long time to run, for this reason some steps have been commented out

'''

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import sys
from numpy import set_printoptions
from progressbar import progressbar
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

# uncomment to show every output rows
pd.set_option('display.max_rows', 100)

# uncomment to show every output columns
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 100)

# how the floating numbers are shown in the numpy arrays
set_printoptions(precision=3) 

# how the floating numbers are shown in pandas
pd.options.display.float_format = '{:,.3f}'.format  

np.random.seed(123)


In [18]:
''' Functions ''' 

def standardize_df(X):
    '''
    Standardize a dataframe
    :param X: a multiindex dataframe
    :return: a standardized dataframe
    '''
    standardizer = StandardScaler().fit(X)
    standardizedX = standardizer.transform(X)

    return standardizedX

def scale_df(X):
    '''
    Scale a dataframe
    :param X: a multiindex dataframe
    :return: a standardized dataframe
    '''
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaledX = scaler.fit_transform(X)

    return scaledX

def write_to_csv(df, title):
    '''
    Dump a multiindex dataframe into an csv file
    :param df: a multiindex dataframe
    :param title: the file title    
    '''
    csv_data = df.to_csv('%s.csv' % title, index=True)

    return csv_data

def write_to_xlxs(df, title):
    '''
    Dump a multiindex dataframe into an xlsx file
    :param df: a multiindex dataframe
    :param title: the file title
    '''
    # Create a Pandas Excel writer using XlsxWriter as the engine.
    writer = pd.ExcelWriter('%s.xlsx' % title, engine='xlsxwriter')

    # Convert the dataframe to an XlsxWriter Excel object.
    df.to_excel(writer, sheet_name='MyData')

    # Close the Pandas Excel writer and output the Excel file.
    writer.save()

def create_multiindex_dataframe(filename, index_list):
    '''
    Create and return a multiindex dataframe from a csv or xlsx file
    :param filename: a csv or xlsx file
    :param index_list: number of column levels, usually [0, 1]
    :return: a multiindex dataframe
    '''

    if "xlsx" in filename:
        df = pd.read_excel(filename, header=index_list, index_col=0)
        df = pd.DataFrame(df)
    elif "csv" in filename:
        df = pd.read_csv(filename, header=index_list, index_col=0)
        df = pd.DataFrame(df)

    return df

def concat_two_df(df1, df2):
    """
    Concat two dataframes on axis=0 (rows)
    :param df1: dataframe 1
    :param df2: dataframe 2
    :return: a new dataframe
    """
    df_concatenated = pd.concat([df1, df2], axis=0)

    return df_concatenated

def concat_two_df_along_columns(df1, df2):
    """
    Concat two dataframes on axis=1 (columns)
    :param df1: dataframe 1
    :param df2: dataframe 2
    :return: a new dataframe
    """
    df_concatenated = pd.concat([df1, df2], axis=1)

    return df_concatenated

def sort_df_index(df, axis, inplace):
    '''  
    Sort indexes
    :param df: a dataframe
    :param axis: the axis along which to sort
    :param inplace: if True, perform operation in-place 
    '''
    df.sort_index(axis=axis, inplace=inplace)
    
def preproc_df_step1(dataframe):
    '''
    This is used as a first step of the preprocessing stage to remove empty or duplicated columns
    :param dataframe: a dataframe
    :return: a dataframe with no empty or duplicated columns
    '''

    # remove features with no values
    columnsToDelete = []
    for key, value in dataframe.iteritems():
        if dataframe[key].std() == 0:
            columnsToDelete.append(key)

    df1 = dataframe.drop(columnsToDelete, axis=1)

    # drop columns with less than 1500 non-NaN values
    df2 = df1.dropna(thresh=1500, axis=1)

    # remove duplicated features, keeping only the first one
    df3 = df2.loc[:, ~df2.T.duplicated(keep='first')]

    # remove rows with one or more NaNs
    df4 = df3.dropna(axis=0, how='any')

    return df3

def find_common_subcolumns_in_multiindex_dataframe(df, column_list):
    '''
    Find common feature names for every quarters
    :param df: a multiindex dataframe
    :param column_list: the list of dataframe columns
    :return: a list of columns common to every quarters
    '''
    common_columns = []

    # the common_columns will be a list of sets
    for q in column_list:
        cols = set(df[q].columns.values.tolist())
        common_columns.append(cols)

    result = common_columns[0]
    for s in common_columns[1:]:
        result.intersection_update(s)

    return list(result)


In [19]:
''' Variables '''

index_list = [0, 1]

# After inspecting the original files, it turns out that the following 47 features are those containing more data
features_to_train = ('adjusted_close', 'volume', 'accountsPayable', 'capitalExpenditures', 'cash',
                  'changeToAccountReceivables', 'changeToInventory', 'changeToLiabilities', 'changeToNetincome',
                  'commonStock', 'costOfRevenue', 'dividendsPaid', 'extraordinaryItems', 'goodWill',
                  'grossProfit', 'incomeBeforeTax', 'incomeTaxExpense', 'intangibleAssets', 'interestExpense',
                  'inventory', 'longTermDebt', 'minorityInterest', 'netIncome', 'netIncomeApplicableToCommonShares',
                  'netReceivables', 'operatingIncome', 'otherAssets', 'otherCashflowsFromFinancingActivities',
                  'otherCashflowsFromInvestingActivities', 'otherCurrentAssets', 'otherCurrentLiab', 'otherLiab',
                  'preferredStockTotalEquity', 'propertyPlantEquipment', 'researchDevelopment', 'retainedEarnings',
                  'sellingGeneralAdministrative', 'shortTermInvestments', 'totalAssets',
                  'totalCashFromFinancingActivities', 'totalCashFromOperatingActivities',
                  'totalCashflowsFromInvestingActivities', 'totalCurrentLiabilities', 'totalLiab',
                  'totalRevenue', 'totalStockholderEquity', 'treasuryStock')

quarters = ['200003', '200006', '200009', '200012', '200103', '200106', '200109', '200112',
            '200203', '200206', '200209', '200212', '200303', '200306', '200309', '200312',
            '200403', '200406', '200409', '200412', '200503', '200506', '200509', '200512',
            '200603', '200606', '200609', '200612', '200703', '200706', '200709', '200712',
            '200803', '200806', '200809', '200812', '200903', '200906', '200909', '200912',
            '201003', '201006', '201009', '201012', '201103', '201106', '201109', '201112',
            '201203', '201206', '201209', '201212', '201303', '201306', '201309', '201312',
            '201403', '201406', '201409', '201412', '201503', '201506', '201509', '201512',
            '201603', '201606', '201609', '201612', '201603', '201606', '201609', '201612',
            '201803', '201806', '201809', '201812', '201903', '201906', '201909', '201912']


In [20]:
''' 
Create two multi-index dataframes for NASDAQ and NYSE, concatenate on axis 1 technical and 
fundamental data. After this step each dataframe has 103 features, 80 quarters (20 years). 
NYSE contains 759 stocks, NASDAQ 848 
'''

# create 2 variables for technical and fundamental data (NASDAQ)
file_name_tech = "nasdaq_technical_data_2000_2019.csv"
file_name_fund = "nasdaq_fundamentals_2000_2019.csv"

# create 2 dataframes
df_tech = pd.read_csv(file_name_tech, header=index_list, index_col=0)
df1_tech = pd.DataFrame(df_tech)

df_fund = pd.read_csv(file_name_fund, header=index_list, index_col=0)
df1_fund = pd.DataFrame(df_fund)

# extract the column names into lists
columns_1 = df1_tech.columns.values.tolist()
columns_2 = df1_fund.columns.values.tolist()

# extract the level(0) column names
column_index_tech_fund_first_level = []
for i in columns_1:
    if i not in column_index_tech_fund_first_level:
        column_index_tech_fund_first_level.append(i[0])
for i in columns_2:
    if i not in column_index_tech_fund_first_level:
        column_index_tech_fund_first_level.append(i[0])

# unique values
column_index_first = np.unique(column_index_tech_fund_first_level)

# inner join the 2 dataframes: they will be just appended on axis=1
tmp_df_nasdaq_complete = df1_tech.join(df1_fund, how='inner')

# this is a workaround to merge tech and fund data, could not find another way
# extract all the columns having level(0) == '200003', creating the main dataframe
first_quarter = column_index_first[0]
data_to_merge = tmp_df_nasdaq_complete.iloc[:, tmp_df_nasdaq_complete.columns.get_level_values(0) == first_quarter]

# loop through the level(0) columns and concat every block to the main one
for q in column_index_first[1:]:
    df_to_add = tmp_df_nasdaq_complete.iloc[:, tmp_df_nasdaq_complete.columns.get_level_values(0) == '%s' % q]
    data_to_merge = pd.concat([data_to_merge, df_to_add], axis=1)


In [21]:
''' 
Repeat the same for NYSE 
'''

file_name_tech = "nyse_technical_data_2000_2019.csv"
file_name_fund = "nyse_fundamentals_2000_2019.csv"


df_tech = pd.read_csv(file_name_tech, header=index_list, index_col=0)
df1_tech = pd.DataFrame(df_tech)

df_fund = pd.read_csv(file_name_fund, header=index_list, index_col=0)
df1_fund = pd.DataFrame(df_fund)

columns_1 = df1_tech.columns.values.tolist()
columns_2 = df1_fund.columns.values.tolist()

column_index_tech_fund_first_level = []
for i in columns_1:
    if i not in column_index_tech_fund_first_level:
        column_index_tech_fund_first_level.append(i[0])
for i in columns_2:
    if i not in column_index_tech_fund_first_level:
        column_index_tech_fund_first_level.append(i[0])

column_index_first = np.unique(column_index_tech_fund_first_level)

tmp_df_nyse_complete = df1_tech.join(df1_fund, how='inner')

first_quarter = column_index_first[0]
data_to_merge = tmp_df_nyse_complete.iloc[:, tmp_df_nyse_complete.columns.get_level_values(0) == first_quarter]

for q in column_index_first[1:]:
    df_to_add = tmp_df_nyse_complete.iloc[:, tmp_df_nyse_complete.columns.get_level_values(0) == '%s' % q]
    data_to_merge = pd.concat([data_to_merge, df_to_add], axis=1)


In [22]:
''' Fill the gaps: The four steps require long time, then they have been commented out.

step 1 investigate the data manually to check the issues and the best way to proceed; the df will be split into several 
       files containing the quarters for each feature, this way it will be easier to fill the gaps
step 2 if a field in a quarter contains data only for Q12, while Q03, Q06, Q09 are empty, replace the blanks with Q12
step 3 replace 0s with None
step 4 count the blanks in the df, if it's < 5000 (less than 4%) then with a first scan the remaining gaps are replaced 
       with the previous value in the row (if there is a value), with the second scan the blank is replaced with the 
       following value; this way the features with too many blanks will be excluded in the following stages


For NYSE the features with no blanks after the previous steps are: 
adjusted_close, volume, accountsPayable, capitalExpenditures, cash, commonStock, costOfRevenue, extraordinaryItems, 
grossProfit, incomeBeforeTax, incomeTaxExpense, interestExpense, minorityInterest, netIncome, 
netIncomeApplicableToCommonShares, operatingIncome, researchDevelopment, sellingGeneralAdministrative, 
totalAssets, totalLiab, totalRevenue, totalStockholderEquity

For NASDAQ: 
adjusted_close, volume, accountsPayable, capitalExpenditures, commonStock, costOfRevenue, grossProfit, incomeBeforeTax,
incomeTaxExpense, interestExpense, netIncome, netIncomeApplicableToCommonShares, operatingIncome, otherAssets,
propertyPlantEquipment, sellingGeneralAdministrative, totalAssets, totalLiab, totalRevenue, totalStockholderEquity

'''


" Fill the gaps: The four steps require long time, then they have been commented out.\n\nstep 1 investigate the data manually to check the issues and the best way to proceed; the df will be split into several \n       files containing the quarters for each feature, this way it will be easier to fill the gaps\nstep 2 if a field in a quarter contains data only for Q12, while Q03, Q06, Q09 are empty, replace the blanks with Q12\nstep 3 replace 0s with None\nstep 4 count the blanks in the df, if it's < 5000 (less than 4%) then with a first scan the remaining gaps are replaced \n       with the previous value in the row (if there is a value), with the second scan the blank is replaced with the \n       following value; this way the features with too many blanks will be excluded in the following stages\n\n\nFor NYSE the features with no blanks after the previous steps are: \nadjusted_close, volume, accountsPayable, capitalExpenditures, cash, commonStock, costOfRevenue, extraordinaryItems, \n

In [23]:
''' 
Regenerate the dataframe concatenating the features previously split in different files and 
concatenate NYSE and NASDAQ in a single file:
    
- step 1, starting from the files in feature_gap_count create a dataframe with only 
the first features (adjusted close);
- step 2, concatenate the previous dataframes along the rows
- step 3, loop through the other features, create temp dataframes and concatenate them to the main dataframe
- step 4, sort the indexes
    
'''

# 47 features are those that after a first scan show a low percentage of NaNs
features = ('adjusted_close', 'volume', 'accountsPayable', 'capitalExpenditures', 'cash',
            'changeToAccountReceivables', 'changeToInventory', 'changeToLiabilities', 'changeToNetincome',
            'commonStock', 'costOfRevenue', 'dividendsPaid', 'extraordinaryItems', 'goodWill',
            'grossProfit', 'incomeBeforeTax', 'incomeTaxExpense', 'intangibleAssets', 'interestExpense',
            'inventory', 'longTermDebt', 'minorityInterest', 'netIncome', 'netIncomeApplicableToCommonShares',
            'netReceivables', 'operatingIncome', 'otherAssets', 'otherCashflowsFromFinancingActivities',
            'otherCashflowsFromInvestingActivities', 'otherCurrentAssets', 'otherCurrentLiab', 'otherLiab',
            'preferredStockTotalEquity', 'propertyPlantEquipment', 'researchDevelopment', 'retainedEarnings',
            'sellingGeneralAdministrative', 'shortTermInvestments', 'totalAssets',
            'totalCashFromFinancingActivities', 'totalCashFromOperatingActivities',
            'totalCashflowsFromInvestingActivities', 'totalCurrentLiabilities', 'totalLiab',
            'totalRevenue', 'totalStockholderEquity', 'treasuryStock')

# step 1
file_folder = "feature_gap_count/"
first_feature = features[0]
file_nyse = "{}nyse_only_{}.xlsx".format(file_folder, first_feature)
file_nasdaq = "{}nasdaq_only_{}.xlsx".format(file_folder, first_feature)

df_nyse = create_multiindex_dataframe(file_nyse, index_list)
df_nasdaq = create_multiindex_dataframe(file_nasdaq, index_list)

# step 2
df = concat_two_df(df_nyse, df_nasdaq)

# step 3
for f in progressbar(features[1:]):
    file_nyse_to_concat = "{}nyse_only_{}.xlsx".format(file_folder, f)
    file_nasdaq_to_concat = "{}nasdaq_only_{}.xlsx".format(file_folder, f)

    df_nyse_to_concat = create_multiindex_dataframe(file_nyse_to_concat, index_list)
    df_nasdaq_to_concat = create_multiindex_dataframe(file_nasdaq_to_concat, index_list)

    df_to_concat = concat_two_df(df_nyse_to_concat, df_nasdaq_to_concat)

    df1 = concat_two_df_along_columns(df, df_to_concat)

    df = df1

# step 4
sort_df_index(df1, 1, True)
sort_df_index(df1, 0, True)


100% (46 of 46) |########################| Elapsed Time: 0:01:16 Time:  0:01:16


In [24]:
''' 
In the following 8 steps drop empty or duplicated features, create the three classes,
scale and standardize data

- step 1: use only columns or duplicated columns 
- step 2: after dropping empty/duplicated features, the quarters have different features, 
then use the quarter with fewer columns to establish the columns to use for every quarter

'''

# step 1

df_with_no_gaps = df1 # this will be used in step 3
df_with_no_gaps_after_preproc1 = preproc_df_step1(df1)  # this will be used only to compute the common features

# extract columns and tickers
column_tuples = df_with_no_gaps_after_preproc1.columns.values.tolist()
tickers = df_with_no_gaps_after_preproc1.index.values.tolist()

# extract the level(0) column names
column_level_zero = []
for i in column_tuples:
    if i not in column_level_zero:
        column_level_zero.append(i[0])

# there are duplicated features, use the following 2 lines to drop them
column_level_zero = np.unique(column_level_zero)  # this is a numpy array
column_level_zero = column_level_zero.tolist()

# step 2
# Uncomment this code to print the quarters having less features
#for q in column_level_zero:
#    cols = df_with_no_gaps_after_preproc1[q].columns.values.tolist()
#    print(q + " : " + str(len(cols)))
#    print(cols)

# Use this code to compute the features common to every quarters
common_columns = find_common_subcolumns_in_multiindex_dataframe(df_with_no_gaps_after_preproc1, column_level_zero)

# 21 features are common to every quarters:
# ['adjusted_close', 'volume', 'totalRevenue', 'totalLiab', 'totalAssets', 'otherAssets', 'totalStockholderEquity', 
# 'capitalExpenditures', 'incomeBeforeTax', 'researchDevelopment', 'incomeTaxExpense', 'netIncome', 
# 'propertyPlantEquipment', 'netIncomeApplicableToCommonShares', 'sellingGeneralAdministrative', 'costOfRevenue', 
# 'grossProfit', 'accountsPayable', 'operatingIncome', 'interestExpense', 'commonStock']


In [25]:
''' 

- step 3 create a new df with only the first 15 years and the 21 common features, rename 
the columns as t0, t1... 
- step 4 extract the columns to be compared, add the calculated columns and drop 
those from T40 to T60, then add the 3 classes
- step 5 three tickers in the train dataset have a certain number of 0s, drop them 

'''

# step 3
quarter_list_first_fifteen_years = column_level_zero[0:60]  # 60 quarters or 15 years

# create a df with the tuples (quarters and common_columns)
df_with_no_gaps_15_years = df_with_no_gaps.loc[:, (quarter_list_first_fifteen_years, common_columns)]

# the columns at level(0) will be renamed to t0, t1, t2 and so on
new_columns = ["t0", "t1", "t2", "t3", "t4", "t5", "t6", "t7", "t8", "t9", "t10", "t11", "t12", "t13", "t14", "t15",
               "t16", "t17", "t18", "t19", "t20", "t21", "t22", "t23", "t24", "t25", "t26", "t27", "t28", "t29", "t30",
               "t31", "t32", "t33", "t34", "t35", "t36", "t37", "t38", "t39", "t40", "t41", "t42", "t43", "t44", "t45",
               "t46", "t47", "t48", "t49", "t50", "t51", "t52", "t53", "t54", "t55", "t56", "t57", "t58", "t59"]

df_with_no_gaps_15_years.columns.set_levels(new_columns, level=0, inplace=True)

# step 4
# t39 = 200912, t43 = 201012, t51 = 201212, t59 = 201412
# the classes will be based on the increase/decrease between the value of the stock at t39 and those after 
# 1 year (t43), 3 years (t51) and 5 years (t59)  
column_t39 = df_with_no_gaps_15_years['t39', 'adjusted_close'].values.tolist()
column_t43 = df_with_no_gaps_15_years['t43', 'adjusted_close'].values.tolist()
column_t51 = df_with_no_gaps_15_years['t51', 'adjusted_close'].values.tolist()
column_t59 = df_with_no_gaps_15_years['t59', 'adjusted_close'].values.tolist()

after_one_year = []
after_three_year = []
after_five_year = []

# compute the increase/decrease after  1, 3, 5 years
for i in range(len(column_t43)):
    value = round(((column_t43[i] - column_t39[i]) / column_t39[i]) * 100, 3)
    after_one_year.append(value)

for i in range(len(column_t51)):
    value = round(((column_t51[i] - column_t39[i]) / column_t39[i]) * 100, 3)
    after_three_year.append(value)

for i in range(len(column_t59)):
    value = round(((column_t59[i] - column_t39[i]) / column_t39[i]) * 100, 3)
    after_five_year.append(value)

# Append the calculated columns
# The class is 1 if the increase is >= 20%, >= 40% or >=100% (after 1, 3, 5 years) otherwise 0
df_with_no_gaps_15_years['Output t43', 'Adj Close'] = df_with_no_gaps_15_years['t43', 'adjusted_close']
df_with_no_gaps_15_years['Output t43', 'Increase % after 1 year'] = after_one_year
df_with_no_gaps_15_years['Output t43', 'Class 1'] = [1 if x >= 20 else 0 for x in after_one_year]

df_with_no_gaps_15_years['Output t51', 'Adj Close'] = df_with_no_gaps_15_years['t51', 'adjusted_close']
df_with_no_gaps_15_years['Output t51', 'Increase % after 3 year'] = after_three_year
df_with_no_gaps_15_years['Output t51', 'Class 2'] = [1 if x >= 40 else 0 for x in after_three_year]

df_with_no_gaps_15_years['Output t59', 'Adj Close'] = df_with_no_gaps_15_years['t59', 'adjusted_close']
df_with_no_gaps_15_years['Output t59', 'Increase % after 5 year'] = after_five_year
df_with_no_gaps_15_years['Output t59', 'Class 3'] = [1 if x >= 100 else 0 for x in after_five_year]

# After 1 year the no. of tickers with increase >= 20% is 744
# After 3 year the no. of tickers with increase >= 40% is 764
# After 5 year the no. of tickers with increase >= 100% is 737
# This means that the classes are almost perfectly balanced

# Uncomment to print the total number of 1 for each class
#print(df_with_no_gaps_15_years.loc[:, ("Output t43", "Class 1")].sum())
#print(df_with_no_gaps_15_years.loc[:, ("Output t51", "Class 2")].sum())
#print(df_with_no_gaps_15_years.loc[:, ("Output t59", "Class 3")].sum())

# Drop the years following 2014, they won't be used
columns_to_drop = new_columns[40:60]
df_with_classes = df_with_no_gaps_15_years.drop(columns_to_drop, axis=1, level=0)

# this file is df_with_no_gaps in step 1
filename = "dataset_with_classes.csv"
df_with_classes = create_multiindex_dataframe(filename, index_list)

# step 5
df_with_classes = df_with_classes.drop(['A', 'AAPL', 'AAWW'])

print(df_with_classes.isna().sum().sum())
print(df_with_classes.shape)

0
(1555, 849)


In [26]:
''' 

- step 7 scale and standardize train and test datasets  
- step 8 store dataframe into csv as dataset_full

'''

# step7

# the 40 quarters to train (10 years)
quarters_to_train = new_columns[0:40]
# the common features as in Preprocessing - step 2
features_to_train = common_columns
# the 997 tickers from the train dataset having no gaps
##tickers_to_train = df_with_classes_train.index.values.tolist()
##tickers_to_test = df_with_classes_test.index.values.tolist()
tickers_full = df_with_classes.index.values.tolist()

# Before training the model scale or standardize the data

# separate input and output
X_full = df_with_classes.loc[:, (quarters_to_train, features_to_train)]
Y1_full = df_with_classes.loc[:, ( "Output t43", "Class 1" )]
Y2_full = df_with_classes.loc[:, ( "Output t51", "Class 2" )]
Y3_full = df_with_classes.loc[:, ( "Output t59", "Class 3" )]

# fit and transform train data
scaled_df = scale_df(X_full)
standardized_df = standardize_df(scaled_df)

x_full = pd.DataFrame(data=standardized_df,
                       index=tickers_full,
                       columns=pd.MultiIndex.from_product([quarters_to_train, features_to_train]))

# step 8

# concatenate the classes along the columns for the train dataset
Y_full = concat_two_df_along_columns(Y1_full, Y2_full)
Y_full = concat_two_df_along_columns(Y_full, Y3_full)

# concatenate Y to X 
# uncomment to store to csv
dataset_full = concat_two_df_along_columns(x_full, Y_full)
#title = "dataset_full"
#write_to_csv(dataset_full, title)
